# Reasoning in Large Language Models:
## A Meta-Analysis of Reinforcement Learning and Test-Time Scaling Approaches
*A comparative review of four 2025 papers on incentivizing and scaling reasoning in LLMs*


## 1. Introduction

Large language models (LLMs) are neural networks, typically built on the transformer architecture, trained on vast text corpora to predict and generate language. Beyond fluent text generation, a major recent research frontier is teaching LLMs to reason: to work through multi-step problems in mathematics, coding, and logic rather than pattern-matching to a single-step answer. This capability, often surfaced through extended "chain-of-thought" generation, has become the defining research theme of 2025, driven by the public release of OpenAI's o1 model and the wave of open replications and analyses that followed.

This report presents a meta-analysis of four recent papers connected by a single theme: how to elicit and scale reasoning ability in LLMs using reinforcement learning (RL) and test-time compute, rather than relying purely on larger pretraining runs or dense human supervision. The selected papers approach this problem from complementary angles: one demonstrates that reasoning can be incentivized through large-scale RL with minimal supervision; one scales that RL recipe further, including to long-context and multimodal settings; one shows that a comparable effect can be achieved far more cheaply through small-scale supervised fine-tuning plus inference-time control; and one critically re-examines the RL training recipe itself, uncovering and correcting a subtle optimization bias. Together they offer a rounded picture of a fast-moving subfield: what works, what is still poorly understood, and where the open questions lie.

**Papers analyzed**

1. DeepSeek-AI et al. (2025). "DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning." arXiv:2501.12948 (published in *Nature*, 2025). [Link](https://arxiv.org/abs/2501.12948)
2. Kimi Team et al. (2025). "Kimi k1.5: Scaling Reinforcement Learning with LLMs." arXiv:2501.12599. [Link](https://arxiv.org/abs/2501.12599)
3. Muennighoff, N. et al. (2025). "s1: Simple Test-Time Scaling." arXiv:2501.19393 (EMNLP 2025). [Link](https://arxiv.org/abs/2501.19393)
4. Liu, Z. et al. (2025). "Understanding R1-Zero-Like Training: A Critical Perspective." arXiv:2503.20783. [Link](https://arxiv.org/abs/2503.20783)


## 2. Paper Summaries


### 2.1 DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning

**Citation:** DeepSeek-AI (2025). *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning.* arXiv:2501.12948; also published in *Nature*, vol. 645, pp. 633–638 (2025).

**Research problem:** Prior reasoning-focused LLMs relied heavily on large volumes of human-annotated reasoning demonstrations, which are expensive to produce and limit scalability. The authors ask whether reasoning can instead be learned purely from reward signals.

**Proposed solution:** The authors train DeepSeek-R1-Zero by applying large-scale RL directly to the DeepSeek-V3-Base model, skipping supervised fine-tuning (SFT) entirely, using Group Relative Policy Optimization (GRPO) with rule-based rewards (answer correctness and output-format adherence). This produces strong reasoning behavior but with readability problems such as language mixing. To fix this, DeepSeek-R1 adds a small "cold-start" set of curated chain-of-thought examples before RL, plus a multi-stage training pipeline that alternates RL and SFT.

**Main results:** DeepSeek-R1 reaches performance comparable to OpenAI's o1 model on math, coding, and STEM reasoning benchmarks. Notably, the RL process gives rise to emergent behaviors — self-verification, reflection, and strategy revision — without being explicitly trained for them (described by the authors as an "aha moment").

**Datasets, architecture, evaluation:** Base architecture is a Mixture-of-Experts transformer (DeepSeek-V3-Base). Training data includes RL rollouts on verifiable math/code tasks plus a modest cold-start CoT dataset. Evaluation spans MMLU, MMLU-Pro, GPQA Diamond, AIME 2024, Codeforces, LiveCodeBench, and Arena-Hard, among others.


### 2.2 Kimi k1.5: Scaling Reinforcement Learning with LLMs

**Citation:** Kimi Team (2025). *Kimi k1.5: Scaling Reinforcement Learning with LLMs.* arXiv:2501.12599.

**Research problem:** The authors note that although RL is a promising new scaling axis beyond next-token-prediction pretraining, prior published RL efforts had not produced results competitive with state-of-the-art reasoning models. The paper investigates how to make RL scaling actually work in practice.

**Proposed solution:** Kimi k1.5 is trained with a simplified, effective RL framework that deliberately avoids more complex machinery such as Monte Carlo tree search, learned value functions, or process reward models. Key ingredients are long-context scaling (allowing much longer reasoning chains during RL) and an improved policy-optimization method, alongside an infrastructure innovation called partial rollout for handling long reasoning trajectories efficiently.

**Main results:** Kimi k1.5 achieves reasoning performance comparable to o1 across math, code, and multimodal benchmarks, demonstrating that a relatively simple RL recipe can scale effectively when context length and infrastructure are engineered carefully.

**Datasets, architecture, evaluation:** The model is a multimodal LLM trained with RL on math, code, and multimodal data recipes. Evaluation includes AIME, MATH500, LiveCodeBench, and multimodal reasoning benchmarks.


### 2.3 s1: Simple Test-Time Scaling

**Citation:** Muennighoff, N., Yang, Z., Shi, W., Li, X. L., Fei-Fei, L., Hajishirzi, H., Zettlemoyer, L., Liang, P., Candès, E., & Hashimoto, T. (2025). *s1: Simple Test-Time Scaling.* arXiv:2501.19393; presented at EMNLP 2025.

**Research problem:** o1-style test-time scaling — using extra inference-time compute to improve reasoning — was shown to work but OpenAI did not disclose its methodology, prompting many costly replication attempts. The authors ask what the simplest possible approach to test-time scaling looks like.

**Proposed solution:** The authors curate s1K, a small dataset of 1,000 questions paired with reasoning traces, selected for difficulty, diversity, and quality. They then supervise-fine-tune Qwen2.5-32B-Instruct on this small dataset and add "budget forcing" at inference: forcibly cutting off the model's thinking, or extending it by appending the word "Wait" to push the model to double-check and revise its answer.

**Main results:** The resulting model, s1-32B, exceeds o1-preview on competition math benchmarks despite using only 1,000 training examples — several orders of magnitude less data than large-scale RL pipelines such as DeepSeek-R1.

**Datasets, architecture, evaluation:** Base model is Qwen2.5-32B-Instruct, fine-tuned with the s1K dataset. Evaluated on AIME24, MATH500, and GPQA-Diamond, with ablations on dataset selection criteria and budget-forcing strategies.


### 2.4 Understanding R1-Zero-Like Training: A Critical Perspective

**Citation:** Liu, Z. et al. (2025). *Understanding R1-Zero-Like Training: A Critical Perspective.* arXiv:2503.20783.

**Research problem:** Following the R1-Zero-style recipe of applying RL directly to a base model without SFT, the authors investigate what actually drives the observed gains, and whether the GRPO optimization method used has hidden biases.

**Proposed solution:** Through controlled experiments across base models (including DeepSeek-V3-Base and Qwen2.5), the authors find that some "aha moment" behavior is already latent in certain base models before RL, suggesting pretraining, not RL alone, seeds some reasoning capacity. They also identify an optimization bias in GRPO that artificially inflates response length, particularly for incorrect answers, and propose Dr. GRPO, a bias-corrected variant.

**Main results:** Dr. GRPO improves token efficiency relative to standard GRPO while maintaining reasoning performance, and the paper's base-model analysis complicates simple narratives about where reasoning ability originates.

**Datasets, architecture, evaluation:** The study reuses R1-Zero-style RL pipelines on multiple open base models (DeepSeek-V3-Base, Qwen2.5 variants). Evaluation focuses on AIME24 and controlled probes of base-model reasoning behavior and response-length dynamics.


## 3. Comparative Analysis

The table below compares the four papers across objectives, architectures, training strategy, evaluation, and reproducibility.

| Aspect | DeepSeek-R1 | Kimi k1.5 | s1 (Test-Time Scaling) | Dr. GRPO Analysis |
|---|---|---|---|---|
| **Core objective** | Elicit reasoning via large-scale RL, minimal human labels | Scale RL end-to-end incl. long context and multimodal input | Match RL-trained reasoners with far less compute via SFT + inference control | Explain and correct biases in RL training dynamics |
| **Base model** | DeepSeek-V3-Base (MoE, ~671B total params) | In-house Kimi base model (multimodal) | Qwen2.5-32B-Instruct | Qwen2.5 and DeepSeek-V3-Base variants |
| **Method type** | Reinforcement learning (GRPO), rule-based rewards | Reinforcement learning, simplified policy optimization | Supervised fine-tuning + inference-time "budget forcing" | Diagnostic study + bias-corrected RL objective (Dr. GRPO) |
| **Training data** | R1-Zero: RL only; R1: small "cold-start" CoT set + RL | Large-scale RL rollouts, long-context + multimodal prompts | s1K: 1,000 curated question-trace pairs | Reuses R1-Zero-style pipelines on multiple base models |
| **Key mechanism** | Verifiable rule-based rewards (answer/format checks) | Long-context scaling, partial rollouts, simplified RL loop | "Budget forcing" appends/truncates thinking tokens at inference | Removes GRPO's length-normalization bias (Dr. GRPO objective) |
| **Main evaluation** | AIME, MATH, Codeforces, GPQA, MMLU, LiveCodeBench | AIME, MATH500, LiveCodeBench, multimodal benchmarks | AIME24, MATH500, GPQA-Diamond | AIME24, base-model reasoning probes, response-length analysis |
| **Headline result** | Comparable to OpenAI o1 on reasoning benchmarks | Comparable to o1 on math/code with a simplified RL recipe | s1-32B exceeds o1-preview on competition math with 1K examples | Identifies and removes an artificial response-length inflation bias in GRPO |
| **Reproducibility** | Weights and R1-Zero/R1 fully open-sourced | Technical report open, some infra details proprietary | Fully open data, code, and budget-forcing recipe (very lightweight) | Fully open code; explicitly designed to be replicated on open models |

**Key comparative observations**

- Objectives diverge along a spectrum from pure RL (DeepSeek-R1-Zero, Kimi k1.5) to pure SFT plus inference control (s1), with DeepSeek-R1 and Dr. GRPO occupying a middle, hybrid or diagnostic position.
- All four papers converge on verifiable, checkable signals — correct math answers, passing test cases, or curated high-quality traces — rather than large opaque human-preference datasets, reflecting a broader move away from RLHF-style reward modeling for reasoning tasks specifically.
- Compute is invested very differently: DeepSeek-R1 and Kimi k1.5 spend heavily on RL rollouts at training time, while s1 shifts much of that cost to inference time via budget forcing, and Dr. GRPO spends compute on careful ablation rather than scale.
- Reproducibility is uneven. s1 is the most lightweight and easiest to reproduce (1,000 examples, open code). DeepSeek-R1 is fully open-weighted but computationally expensive to retrain. Kimi k1.5's infrastructure (e.g., partial rollout systems) is less accessible to reproduce outside large labs.


## 4. Insights and Reflection

Reading these four papers together reveals a field that moved extremely quickly in a narrow window (December 2024–March 2025) once a workable RL recipe for reasoning became public, and that is now in a phase of both scaling up and critically re-examining the methods that emerged.

| Trend | Evidence across papers |
|---|---|
| Shift from imitation to incentive | DeepSeek-R1 and Kimi k1.5 both show reasoning skills emerging from reward signals rather than being copied from human demonstrations. |
| Verifiable rewards over learned reward models | R1 and Kimi k1.5 favor rule-based, checkable rewards (correct answer, valid format) to avoid reward hacking; s1 sidesteps reward modeling entirely by using supervised traces. |
| Compute can move to inference time | s1's budget forcing and R1's long chain-of-thought both show that letting a model "think longer" at inference is a cheap, effective lever, complementing training-time scaling. |
| RL recipes were not yet well understood | Dr. GRPO shows the popular GRPO objective has a hidden bias that inflates response length; this kind of correction was only possible once R1-style pipelines became public and reproducible. |
| Small curated data can rival massive RL runs | s1K's 1,000 examples produce a model competitive with far larger RL-trained systems, challenging the assumption that reasoning gains require massive rollout budgets. |

**Most promising directions**

Test-time scaling (as in s1) stands out as unusually promising precisely because of its low cost: it decouples reasoning gains from massive RL infrastructure, making strong reasoning more accessible outside large industrial labs. At the same time, the Dr. GRPO paper suggests that some of the excitement around emergent RL behaviors needs tempering — gains attributed to RL may partly reflect artifacts of the optimization objective or capabilities already latent in the base model.

**Common challenges acknowledged**

- Reward hacking and gaming remain a persistent risk when using RL with either rule-based or learned rewards, motivating the shift toward simple, verifiable rewards over neural reward models.
- Readability and language consistency of reasoning traces is a recurring problem for pure RL training (seen clearly in DeepSeek-R1-Zero), addressed only through additional supervision or reward terms.
- Attributing gains correctly is difficult: it is often unclear how much of a model's reasoning improvement comes from RL itself versus the base model's pretraining, versus incidental biases in the optimization algorithm, as Dr. GRPO demonstrates.
- Compute and infrastructure costs for large-scale RL (long rollouts, long context) remain a barrier to reproducibility, even when weights or reports are shared publicly.

**Future research directions**

- Combining cheap test-time scaling techniques (budget forcing, extended thinking) with lighter-weight RL to find better cost/performance trade-offs.
- Developing standardized diagnostics, akin to Dr. GRPO's analysis, to separate genuine reasoning gains from optimization artifacts across future RL recipes.
- Extending these techniques beyond math and code to domains with less easily verifiable rewards, such as open-ended writing or scientific hypothesis generation.
- Better understanding of what pretraining contributes to latent reasoning capacity, to guide where RL or fine-tuning effort is best spent.


## 5. Conclusion

This meta-analysis examined four 2025 papers united by a common theme: eliciting and scaling reasoning capability in LLMs through reinforcement learning and test-time compute rather than through larger pretraining alone. DeepSeek-R1 and Kimi k1.5 demonstrate that large-scale RL with simple, verifiable rewards can produce reasoning performance competitive with proprietary systems like OpenAI's o1. s1 shows that a remarkably small, carefully curated dataset combined with a simple inference-time control mechanism can achieve similar effects at a fraction of the cost. Understanding R1-Zero-Like Training complicates the picture further, showing that part of what looked like an RL-driven breakthrough is entangled with base-model pretraining effects and a correctable bias in the GRPO objective itself.

Taken together, these papers show a field evolving from a single striking demonstration (RL can incentivize reasoning) toward a more nuanced, mechanistic understanding of why these methods work, and toward increasingly efficient and accessible recipes for achieving similar results. The likely trajectory is one of continued convergence between training-time RL and test-time compute strategies, paired with more rigorous ablation work of the kind Dr. GRPO exemplifies, as the community moves from headline benchmark wins toward a durable, well-understood toolkit for building reasoning-capable LLMs.


## References

1. DeepSeek-AI (2025). DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning. arXiv:2501.12948. https://arxiv.org/abs/2501.12948

2. Kimi Team (2025). Kimi k1.5: Scaling Reinforcement Learning with LLMs. arXiv:2501.12599. https://arxiv.org/abs/2501.12599

3. Muennighoff, N. et al. (2025). s1: Simple Test-Time Scaling. arXiv:2501.19393. https://arxiv.org/abs/2501.19393

4. Liu, Z. et al. (2025). Understanding R1-Zero-Like Training: A Critical Perspective. arXiv:2503.20783. https://arxiv.org/abs/2503.20783
